In [ ]:
import yaml
from pathlib import Path
import time
import pandas as pd

from fefetlab.instruments.visa_session import VisaConfig, VisaSession
from fefetlab.b1500 import B1500, B1500Error

# 1. 读取配置
with open("configs/instruments.yaml", 'r', encoding='utf-8') as f:
    inst_cfg = yaml.safe_load(f)['b1500']
with open("configs/channel_map.yaml", 'r', encoding='utf-8') as f:
    roles = yaml.safe_load(f)['current_device']['role_map']

# 2. 构造 VisaConfig 和通道号
visa_cfg = VisaConfig(
    resource=inst_cfg['resource'],
    timeout_ms=inst_cfg['timeout_ms'],
    write_termination=inst_cfg['write_termination'],
    read_termination=inst_cfg['read_termination'],
    send_end=inst_cfg['send_end']
)

CH_G = roles['G']
CH_D = roles['D']
CH_S = roles['S']

# 3. 扫描参数：固定的 Vg 与 Vs，扫描的 Vd 列表
vg_points = [0.0, -0.4, -0.8]      # 根据 08 的结果挑选代表性 Vg
vd_points = [0.0, 0.05, 0.10]      # 扫描的 Vd 列表（单位 V）
vs_fixed  = 0.0                    # 源极固定为 0 V

VRANGE_G = 0                       # 根据仪器设定的电压量程索引
VRANGE_D = 0
VRANGE_S = 0
I_COMP   = 1e-3                    # 电流合规值 1 mA
delay_s  = 0.3                     # 设置后等待时间

rows = []

def measure_point(b1500: B1500, vg: float, vd: float, vs: float) -> dict:
    """给定 Vg/Vd/Vs 施压并测量电流，返回结果字典。"""
    result = {"vg_set": vg, "vd_set": vd, "vs_set": vs}
    try:
        b1500.dv(CH_G, VRANGE_G, vg, I_COMP)
        b1500.dv(CH_D, VRANGE_D, vd, I_COMP)
        b1500.dv(CH_S, VRANGE_S, vs, I_COMP)
        time.sleep(delay_s)
        result["ig_A"] = b1500.ti(CH_G)
        result["id_A"] = b1500.ti(CH_D)
        result["is_A"] = b1500.ti(CH_S)
        result["err"]  = b1500.errx()
    except B1500Error as e:
        result.update({"ig_A": None, "id_A": None, "is_A": None, "err": str(e)})
    finally:
        b1500.dz([CH_G, CH_D, CH_S])
        b1500.cl([CH_G, CH_D, CH_S])
        time.sleep(0.1)
    result["timestamp"] = time.time()
    return result

# 4. 建立会话并执行扫描
with VisaSession(visa_cfg) as session:
    b = B1500(session)
    b.fmt(5)
    b.av(10, 1)
    b.fl(0)
    b.cn([CH_G, CH_D, CH_S])
    for vg in vg_points:
        for vd in vd_points:
            rows.append(measure_point(b, vg, vd, vs_fixed))

# 5. 保存数据
df = pd.DataFrame(rows)
run_dir = Path("runs") / time.strftime("%Y%m%d_%H%M%S_id_vd_sweep")
run_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(run_dir / "parsed.csv", index=False, encoding="utf-8-sig")
df.to_json(run_dir / "raw.json", orient="records", force_ascii=False, indent=2)

# QC 标记：根据 err 是否为 0，以及电流是否有效
qc_records = []
for _, row in df.iterrows():
    issues = []
    if str(row["err"]).strip() not in ["0", '0,"No Error."', "0 (No Error)"]:
        issues.append("instrument_error")
    if row["id_A"] is None:
        issues.append("missing_data")
    qc_records.append({
        "vg_set": row["vg_set"],
        "vd_set": row["vd_set"],
        "status": "ok" if not issues else "suspect",
        "issues": ";".join(issues),
    })
pd.DataFrame(qc_records).to_csv(run_dir / "qc.csv", index=False, encoding="utf-8-sig")
print(f"数据保存在 {run_dir}")